In [ ]:
from pathlib import Path
import json
import warnings
import sys
import pickle

import numpy as np
import pandas as pd
import h5py
from scipy import stats

from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
sns.set_theme(style='ticks')

In [ ]:
repo_dir = Path("../../..")

In [ ]:
if str(repo_dir) not in sys.path:
    sys.path.append(str(repo_dir))
    
from analysis.curve_fitting.src.fitting_functions import LOSS_FUNCTIONS
from analysis.curve_fitting.src.utils import apply_filters, load_yaml, convert_loss_parameters, convert_loss_parameters_batch

from visualization.src.utils import set_ticks, save_figs, apply_hiearchical_aggregation
from visualization.src.utils import BENCHMARK_NAME_MAPPING, BENCHMARK_COLORS, METRICS_COMPACT, GENERIC_GROUPING_COLUMNS, ARCHITECUTURE_FAMILY_COLORS, READOUT_COLORS
from visualization.src.visualize import plot_reg, plot_reg_bivariate, plot_confidence_intervals

In [ ]:
repo_dir = Path("../../..")
artifacts_dir = repo_dir / "extract_n_eval" / "artifacts"
save_dir = repo_dir / "visualization" / "paper" / "main" / "figures"

assert artifacts_dir.exists(), f"artifacts directory {artifacts_dir} does not exist!"

In [ ]:
results_file = artifacts_dir / "mapping_results.csv"
df_all = pd.read_csv(results_file)

In [ ]:
df_all.columns

## Load Experiment Configuration

In [ ]:
analysis_dir = repo_dir / 'analysis'
config_dir = analysis_dir / 'curve_fitting/configs/mapping'
results_dir = analysis_dir / 'curve_fitting/outputs/fitting_results'
# results_dir = analysis_dir / 'curve_fitting/outputs/fitting_results_mapping'

In [ ]:
datasets = [
    'tvsd',
    'nsd',
    'things_fmri',
    'things_eeg1',
    'things_eeg2',
    'things_meg',
]

mapping_types = [
    'linear',
    'attention_individual',
    'attention_shared',
]

METRIC = 'pearsonr_nc'

In [ ]:
all_configs = {}

for ds in datasets:
    for m in mapping_types:
        yaml_config = config_dir / f'{m}/{ds}.yaml'
        all_configs[f"{ds}-{m}"] = load_yaml(yaml_config)

In [ ]:
L_fit_dict = {key: config['fitting_parameters']['loss_function'] for key, config in all_configs.items()}
L_viz_dict = {key: config['visualization']['loss_function'] for key, config in all_configs.items()}
x_scale_dict = {key: float(config['fitting_parameters']['X_scaler']) for key, config in all_configs.items()}



## Apply Data Filters

In [ ]:
all_df = {
    name: apply_filters(df_all, config.get('data_filters', {}))
    for name, config in all_configs.items()
}

## Load Fitting Results

In [ ]:
optimized_params_dict = {}
opt_params_boot_dict = {}

for c in all_configs.keys():
    with open(results_dir / f'mapping-{c}' / 'results.pkl', 'rb') as f:
        results = pickle.load(f)

    L_fit = L_fit_dict[c]
    L_viz = L_viz_dict[c]
    optimized_params_dict[c] = convert_loss_parameters(results['optimized_parameters'], L_fit, L_viz)

    # Convert bootstrapped parameters
    opt_params_boot = results['optimized_parameters_bootstrapped']
    opt_params_boot_dict[c] = convert_loss_parameters_batch(
        params=opt_params_boot,
        src_loss=L_fit,
        dst_loss=L_viz
    )

## Visualize

In [ ]:
x_extend = 15
X_str = r'$$\tilde{F}$$'
X_str = r'$$F$$'
linewidth = 3.0
alpha_scatter = 0.2
alpha_ci = 0.2
alpha_fit = 1.0
fig_multiplier = 0.75
fig_multiplier = 0.6
# fig_multiplier = 0.5
# fig_multiplier = 1.0
figsize = (10, 8)
figsize = (10, 7)
figsize = (10, 6)
# figsize = (10, 5)
# figsize = (22, 10)
figsize = (fig_multiplier * figsize[0], fig_multiplier * figsize[1])


percentile_ci = 95

In [ ]:

from cProfile import label


sns.set_theme(style='whitegrid')
sns.set_theme(style='ticks')


fig, axes = plt.subplots(1, 1, figsize=figsize, dpi=300)
ax = axes
for idx, ds in enumerate(datasets):
    # ax = axes.flatten()[idx]

    for j, m in enumerate(['linear']):
        exp_name = f'{ds}-{m}'
        print(f"Plotting {exp_name}...")
        ### Neuro
        df_region = all_df[exp_name]
        optimized_params_neuro = optimized_params_dict[exp_name]
        opt_params_boot_neuro = opt_params_boot_dict[exp_name]
        L = LOSS_FUNCTIONS[L_viz_dict[exp_name]]
        x_scaler = x_scale_dict[exp_name]
        X = df_region.fitting_samples.values / x_scaler
        
        # color = colors[m]
        color = BENCHMARK_COLORS[ds]
        # color = color_palette[j]
        # alpha_scatter = 1.0
        sns.scatterplot(data=df_region, x='fitting_samples', y='pearsonr_nc', ax=ax, color=color, alpha=alpha_scatter)
        plot_reg(X, optimized_params_neuro, L, ax, color=color, x_extend=2, linestyle='-', X_str=X_str, x_scaler=x_scaler, show_x_scaler=False, linewidth=linewidth, legend=False, alpha=alpha_fit)
        # plot_reg(X, optimized_params_neuro, L, ax, color=color, x_extend=x_extend, linestyle='-', X_str=X_str, x_scaler=x_scaler, show_x_scaler=False, linewidth=linewidth, alpha=alpha_fit)
        plot_confidence_intervals(X, opt_params_boot_neuro, L, ax, color=color, x_scaler=x_scaler, x_extend=2, alpha=0.1, percentile=percentile_ci, invert_y=True)


    ### Formatting
ax.set_xscale('log')
ax.set_xlabel('Neural Samples', fontsize=12, fontweight='bold')
ax.set_ylabel('Alignment Score', fontsize=16, fontweight='bold')
ax.set_ylabel('Within-Dataset Alignment', fontsize=12, fontweight='bold')
ax.set_title('Scaling Linear Readout', fontsize=20, fontweight='bold')
ax.set_title(None)
# ax = set_ticks(ax, xticks_mode='log', yticks_mode=None, yticks=[0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0])
ax = set_ticks(ax, xticks_mode='log', yticks_mode=None, yticks=[0.0, 0.2, 0.4, 0.6, 0.8, 1.0], minor_grid=False)
ax = set_ticks(ax, xticks_mode='log', yticks_mode=None, yticks=[0.0, 0.2, 0.4, 0.6, 0.8, 1.0])

# ### Legend
handles, labels = ax.get_legend_handles_labels()
# labels = [
#     'Linear                      ' + labels[0],
#     'Attention Individual  '  + labels[1],
#     'Attention Shared     '  + labels[2]
# ]
labels = [
    f'{BENCHMARK_NAME_MAPPING[ds]}  ' + label
    for ds, label in zip(datasets, labels)
]
labels = [
    f'{BENCHMARK_NAME_MAPPING[ds]}'
    for ds in datasets
]
ax.legend(handles, labels)

# ax.legend().remove()

ax.spines[['right', 'top']].set_visible(False)
    
plt.tight_layout()


# figures_dir = save_dir
# fig_name = 'fig5_mapping_linear_only'
# formats = ['pdf', 'png', 'svg']
# save_figs(figures_dir, fig_name, formats=formats)

figures_dir = save_dir
fig_name = 'fig6-mapping-linear_only'
formats = ['pdf', 'png', 'svg']
save_figs(fig=fig, save_dir=figures_dir, base_filename=fig_name, dpi=300, formats=formats)


## Table 2

In [ ]:
data = df_all.copy()
data = data[data["data_pct"] == 100]
data = data.groupby(
    ['probe_type', 'probe_name', 'benchmark_name']
).agg(
    {METRIC: 'mean'}
).reset_index()

data_avg = data.groupby(
    ['probe_type', 'probe_name']
).agg(
    {METRIC: 'mean'}
).reset_index()
data_avg['benchmark_name'] = 'Average'

data['benchmark_name'] = data['benchmark_name'].map(BENCHMARK_NAME_MAPPING)
data[METRIC] = data[METRIC].round(3)

data = pd.concat([data, data_avg], axis=0, ignore_index=True)
data = data.pivot_table(
    index=['benchmark_name'],
    columns=['probe_name'],
    values=[METRIC]
)
data
data.to_dict()

In [ ]:
data

In [ ]:
df_all.benchmark_name.unique()